# Portfolio Risk and Crowding Lab

**Question:** how much risk do six book names really diversify, and what changes when capital is reallocated?

Industry/factor returns are real. Books, costs, financing and capital policies are illustrative. Revised public files do not reconstruct historical availability or live hedge-fund results.

## 1. Validate inputs and disclose the books

In [1]:
from pathlib import Path
import os, json
if not Path('research/strategist_lab').exists():
    os.chdir('..')
assert Path('research/strategist_lab').exists(), 'Start Jupyter in the repository root or notebooks folder.'
from research.strategist_lab.data import load_data
from research.strategist_lab import OUTPUT
industries, factors = load_data()
print('Input window:', industries.index[0].date(), 'to', industries.index[-1].date())
print('Rows:', len(industries), '| Public data vintage and SHA-256 hashes are in data/strategist_lab/manifest.json')


Input window: 2015-01-02 to 2025-12-31
Rows: 2766 | Public data vintage and SHA-256 hashes are in data/strategist_lab/manifest.json


In [2]:
import pandas as pd
from research.strategist_lab.portfolio import BOOK_NAMES, BOOK_WEIGHTS, RiskConfig, run_portfolio
from research.strategist_lab.data import INDUSTRIES
print(pd.DataFrame(BOOK_WEIGHTS, index=BOOK_NAMES, columns=INDUSTRIES).to_string())


                    NoDur  Durbl  Manuf  Enrgy  HiTec  Telcm  Shops  Hlth  Utils  Other
Technology tilt      0.00   0.00  -0.20  -0.20    0.5   0.10   0.00   0.0   0.00   0.00
Digital consumer    -0.15   0.10   0.00   0.00    0.3   0.00   0.30   0.0  -0.15   0.00
Defensive spread     0.20  -0.20  -0.20   0.00    0.0   0.00   0.00   0.2   0.20   0.00
Cyclical spread     -0.15   0.00   0.25   0.25    0.0   0.00   0.00   0.0  -0.20   0.15
Consumer relative    0.00   0.25   0.00   0.00    0.0  -0.25   0.25   0.0   0.00  -0.25
Diversified equity   0.10   0.10   0.10   0.10    0.1   0.10   0.10   0.1   0.10   0.10


## 2. Run a chronological capital-allocation experiment

Each decision uses only prior sessions. Monthly rules share position bounds and a turnover budget. Sleeve weights drift between decisions; explicit financing, internal maintenance and allocation costs enter P&L.

In [3]:
metrics, daily, books = run_portfolio(industries, factors, RiskConfig())
print('Evaluation:', metrics['evaluation_start'], 'to', metrics['evaluation_end'])
assert (pd.to_datetime(daily.forecast_max_date) < pd.to_datetime(daily.date)).all()
print(pd.DataFrame(metrics['performance']).to_string(index=False))


Evaluation: 2020-01-02 to 2025-12-31
          method     cagr  annual_vol  excess_sharpe  max_drawdown  total_return  allocation_cost_bps  mean_one_way_turnover
           equal 0.092359    0.082790       0.766627     -0.116340      0.696617             5.332803               0.011735
     inverse_vol 0.082614    0.069700       0.767759     -0.091304      0.608035             6.029858               0.014190
minimum_variance 0.059431    0.038946       0.774975     -0.046381      0.412664             8.067834               0.021366


![Risk-return trade-off](../assets/images/strategist/portfolio-performance.svg)

A reduction in volatility and drawdown is accompanied by lower realised return. The optimiser targets variance, not alpha or Sharpe.

## 3. Reconcile risk and common-factor exposures

In [4]:
import numpy as np
print(pd.DataFrame({'book': BOOK_NAMES, 'capital_weight': metrics['weights'], 'annual_vol_contribution': metrics['risk_contributions_annual']}).to_string(index=False))
print('Euler error:', metrics['risk_reconciliation_error'])
print('Factor-model error:', metrics['factor_model']['reconciliation_error'])
print('Portfolio betas:', dict(zip(metrics['factor_names'], metrics['factor_model']['portfolio_beta'])))
assert abs(sum(metrics['risk_contributions_annual']) - metrics['annual_vol']) < 1e-10


              book  capital_weight  annual_vol_contribution
   Technology tilt        0.172720                 0.011048
  Digital consumer        0.030401                 0.003357
  Defensive spread        0.343577                -0.007492
   Cyclical spread        0.194208                 0.012840
 Consumer relative        0.228895                 0.014696
Diversified equity        0.030199                 0.003790
Euler error: 4.336808689942018e-19
Factor-model error: 7.517417406881916e-21
Portfolio betas: {'Mkt-RF': 0.1637327842888287, 'SMB': -0.008953872616099492, 'HML': -0.036362705373715803, 'RMW': 0.011986651159610453, 'CMA': 0.020975724824058344}


![Book risk contributions](../assets/images/strategist/risk-contributions.svg)

![Return correlation](../assets/images/strategist/correlation.svg)

Return co-movement reveals internal common risk; it does not establish outside funds’ holdings or external crowding.

## 4. Check tail models against actual subsequent losses

Report exception coverage and dependence, including failures. VaR is a quantile estimate, not a guaranteed loss limit.

In [5]:
print(json.dumps(metrics['coverage']['minimum_variance'], indent=2))
print(pd.DataFrame(metrics['historical_stress']).to_string(index=False))


{
  "var95": {
    "observations": 1508,
    "exceptions": 88,
    "expected_exceptions": 75.4,
    "exception_rate": 0.058355437665782495,
    "uc_lr": 2.1083447641901785,
    "uc_p": 0.1464977021975329,
    "independence_lr": 4.192693152022002,
    "independence_p": 0.040598553423999664,
    "conditional_coverage_p": 0.042829894177855425,
    "transitions": [
      [
        1341,
        78
      ],
      [
        78,
        10
      ]
    ]
  },
  "var99": {
    "observations": 1508,
    "exceptions": 27,
    "expected_exceptions": 15.08,
    "exception_rate": 0.01790450928381963,
    "uc_lr": 7.708672986091472,
    "uc_p": 0.00549561363232509,
    "independence_lr": 10.85271793074594,
    "independence_p": 0.0009865085854813874,
    "conditional_coverage_p": 9.32062798275532e-05,
    "transitions": [
      [
        1457,
        23
      ],
      [
        23,
        4
      ]
    ]
  },
  "normal_var99": {
    "observations": 1508,
    "exceptions": 27,
    "expected_exceptio

![VaR exceptions](../assets/images/strategist/var-backtest.svg)

Historical stresses replay final exposures on past windows. They are counterfactual stress estimates, not the allocator’s realised path.

## 5. Explore valid covariance shocks and parameter sensitivity

In [6]:
from research.strategist_lab.portfolio import scenario_risk
cov=np.array(metrics['covariance_daily']); w=np.array(metrics['weights'])
for blend in [0,.5,1.]:
    vol, var99, matrix=scenario_risk(cov,w,vol_multiplier=1.5,correlation_blend=blend,leverage=1.25)
    print(f'Correlation blend {blend:.0%}: annual vol {vol:.2%}, normal one-day VaR {var99:.2%}')
    assert np.linalg.eigvalsh(matrix).min() > -1e-10
print(pd.read_csv(OUTPUT / 'portfolio/lookback_sensitivity.csv').to_string(index=False))
print((OUTPUT / 'portfolio/volatility_ratio_interval.json').read_text())


Correlation blend 0%: annual vol 7.17%, normal one-day VaR 1.05%
Correlation blend 50%: annual vol 16.29%, normal one-day VaR 2.39%
Correlation blend 100%: annual vol 21.89%, normal one-day VaR 3.21%
 lookback           method  volatility     cagr  var99_exception_rate
      126            equal    0.082790 0.092359              0.023210
      126      inverse_vol    0.070432 0.086043              0.021883
      126 minimum_variance    0.038682 0.062437              0.018568
      252            equal    0.082790 0.092359              0.018568
      252      inverse_vol    0.069700 0.082614              0.019231
      252 minimum_variance    0.038946 0.059431              0.017905
      504            equal    0.082790 0.092359              0.015252
      504      inverse_vol    0.068294 0.081851              0.015915
      504 minimum_variance    0.039118 0.059707              0.015252
{
  "quantity": "realised volatility ratio: minimum variance / equal allocation",
  "point": 0.47041

## 6. Decision and limitations

Capital weights and risk contributions are different quantities. Allocation changes exposures, returns and drawdowns together. The 99% tail forecasts warrant further investigation because exceptions are too frequent and clustered in this sample.

Research indices are not directly tradable. A real portfolio process needs reconciled positions, security-level factors, liquidity, borrow and funding, expected alpha and prospective validation. Cost assumptions here cannot estimate capacity. See [full methods](../research/strategist_lab/METHODOLOGY.md) and the [decision brief](../research/strategist_lab/DECISION_BRIEF.md).